# 05 · Dead-Letter Queue (DLQ)

Every queue (and every subscription) has a **sub-queue** for poison messages: `<entity>/$DeadLetterQueue`. Messages land there when:

- `DeliveryCount` exceeds `MaxDeliveryCount` (configured per queue — we set `3` on `demo-dlq`)
- TTL expires (if `DeadLetteringOnMessageExpiration = true`)
- You explicitly call `DeadLetterMessageAsync(reason, description)`

Once in the DLQ, messages stay there until you process them. There's no auto-cleanup.


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.DlqDemoQueueName);

await sender.SendMessageAsync(new ServiceBusMessage("bad-payload")
{
    ApplicationProperties = { ["expectFailure"] = true }
});
Console.WriteLine("Sent a 'bad' message to demo-dlq.");

## 1. Explicitly dead-letter a message

In [ ]:
var receiver = client.CreateReceiver(Config.DlqDemoQueueName);
var msg = await receiver.ReceiveMessageAsync(TimeSpan.FromSeconds(5));

Console.WriteLine($"Got: {msg.Body}  (DeliveryCount={msg.DeliveryCount})");

await receiver.DeadLetterMessageAsync(
    msg,
    deadLetterReason: "ValidationFailed",
    deadLetterErrorDescription: "Payload failed schema validation");

Console.WriteLine("Dead-lettered.");

## 2. Read from the DLQ sub-queue

Use `ServiceBusReceiverOptions.SubQueue = SubQueue.DeadLetter`.


In [ ]:
var dlq = client.CreateReceiver(Config.DlqDemoQueueName, new ServiceBusReceiverOptions
{
    SubQueue = SubQueue.DeadLetter
});

var dead = await dlq.ReceiveMessageAsync(TimeSpan.FromSeconds(5));

Console.WriteLine($"Body              : {dead.Body}");
Console.WriteLine($"DeadLetterReason  : {dead.DeadLetterReason}");
Console.WriteLine($"DeadLetterDescr.  : {dead.DeadLetterErrorDescription}");
foreach (var kv in dead.ApplicationProperties)
    Console.WriteLine($"  app.{kv.Key} = {kv.Value}");

## 3. Resubmit a DLQ message after fixing the upstream

A common pattern is to clone the body, drop the dead-letter system properties, and send back to the main queue.

In [ ]:
var resubmitted = new ServiceBusMessage(dead.Body)
{
    MessageId = dead.MessageId,
    ContentType = dead.ContentType,
    Subject = dead.Subject,
};
foreach (var kv in dead.ApplicationProperties)
    if (!kv.Key.StartsWith("DeadLetter")) resubmitted.ApplicationProperties[kv.Key] = kv.Value;

await sender.SendMessageAsync(resubmitted);
await dlq.CompleteMessageAsync(dead);

Console.WriteLine("Resubmitted and removed from DLQ.");

await dlq.DisposeAsync();
await receiver.DisposeAsync();
await sender.DisposeAsync();
await client.DisposeAsync();

## Tip: monitor DLQ depth

Use the Admin client's `GetQueueRuntimePropertiesAsync()` and check `DeadLetterMessageCount`. Wire it to an Azure Monitor alert in production.

Next: [`06-scheduled-deferred.ipynb`](06-scheduled-deferred.ipynb)
